# Ten useful ways to play

Read [the route guide](06_ten_useful_ways_to_play.md) for the goal, limits,
and next step of each exercise. These are finite teaching exercises, not native
host functional calculations or evidence that an MSS lift is licensed.
Every CHECKED outcome below has only the scope of its stated local rule.
No provider is called and no output directory is created.


In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from fractions import Fraction

here = Path.cwd().resolve()
lesson = here if (here / "educational_governor.py").is_file() else here / "omega_engine_education/documentation/004_oepfp/_how_to_play"
sys.path.insert(0, str(lesson))
from educational_governor import EducationalHostGovernor

FIELDS = {"schema", "carrier", "relation_label", "goal", "candidate"}


def check(packet):
    def refused(reason):
        return {"status": "REFUSED", "reason": reason, "scope": "symbol-game-basic-packet/v1"}

    if type(packet) is not dict or set(packet) != FIELDS:
        return refused("SCHEMA")
    if packet["schema"] != "symbol-game-basic-packet/v1":
        return refused("SCHEMA")
    carrier = packet["carrier"]
    if (
        type(carrier) is not list
        or not carrier
        or any(type(mark) is not str or not mark for mark in carrier)
        or len(set(carrier)) != len(carrier)
    ):
        return refused("CARRIER")
    label = packet["relation_label"]
    if type(label) is not str or not label:
        return refused("RELATION_LABEL")
    goal, candidate = packet["goal"], packet["candidate"]
    if (
        type(goal) is not list
        or len(goal) != 2
        or type(candidate) is not list
        or len(candidate) != 3
        or any(type(value) is not str for value in goal + candidate)
    ):
        return refused("SCHEMA")
    left, relation, right = candidate
    if any(mark not in carrier for mark in goal + [left, right]):
        return refused("OUTSIDE_CARRIER")
    if goal[0] == goal[1] or left == right:
        return refused("SELF_RELATION")
    if relation != label:
        return refused("RELATION_LABEL")
    if [left, right] != goal:
        return refused("GOAL_UNMET")
    return {
        "status": "CHECKED",
        "property": "declared-directed-separation",
        "scope": "symbol-game-basic-packet/v1",
    }


def example(candidate=None):
    return {
        "schema": "symbol-game-basic-packet/v1",
        "carrier": ["a", "b"],
        "relation_label": "separates",
        "goal": ["a", "b"],
        "candidate": ["a", "separates", "b"] if candidate is None else candidate,
    }

def board(fuel=10):
    return EducationalHostGovernor([0,1,2], ['0->1','1->2','2->0'],
                                   {'0->1':2, '1->2':1, '2->0':3}, fuel=fuel)

def cut(host, value):
    return host.execute_move({'move_type':'CUT', 'payload':{'proposed_claims':{'2->0':value}}})


## 1. Distinguish marks

A directed distinction must meet the declared goal.

In [ ]:
receipt = check(example())
assert receipt['status'] == 'CHECKED'
print(receipt)

## 2. Repair a packet

Repair the candidate without silently changing the carrier.

In [ ]:
packet = example(['a','separates','c'])
assert check(packet)['reason'] == 'OUTSIDE_CARRIER'
packet['candidate'][-1] = 'b'
assert check(packet)['status'] == 'CHECKED'
print(check(packet))

## 3. Predict and test

Write a prediction, then compare it with changed-input checks.

In [ ]:
predictions = ['SELF_RELATION','GOAL_UNMET','CHECKED']
candidates = [['a','separates','a'], ['b','separates','a'], ['a','separates','b']]
outcomes = [check(example(c)) for c in candidates]
assert [r.get('reason',r['status']) for r in outcomes] == predictions
print(outcomes)

## 4. Repair a finite graph

These moves change reports; they are not gauge transformations.

In [ ]:
host = board()
defects = [host.compute_total_defect()]
for value in [-1,-3]:
    assert cut(host,value)['decision'] == 'ADMITTED'
    defects.append(host.compute_total_defect())
assert defects == [6,2,0]
print('Triangle circulation defects:', defects)

## 5. Check a certificate

A potential witnesses consistency of this triangle. This is not an optimality certificate.

In [ ]:
reports = {'0->1':Fraction(2),'1->2':Fraction(1),'2->0':Fraction(-3)}
potential = {0:Fraction(0),1:Fraction(2),2:Fraction(3)}
def consistent(p):
    return all(p[v]-p[u] == reports[f'{u}->{v}'] for u,v in [(0,1),(1,2),(2,0)])
assert consistent(potential)
altered = dict(potential); altered[2] += 1
assert not consistent(altered)
print('Valid potential passes; altered potential fails.')

## 6. Refuse an unsafe claim

The teaching model refuses a carrier escape. Missing native ports are exercised separately in demos 8 and 10.

In [ ]:
host = board(); before = dict(host.claims)
receipt = host.execute_move({'move_type':'CUT','payload':{'proposed_claims':{'3->4':0}}})
assert receipt['decision'] == 'REFUSED_CARRIER_DOMAIN'
assert host.claims == before
print(receipt)

## 7. Challenge an argument

Falsify the assertion that reversing a relation still meets the directed goal.

In [ ]:
counterexample = example(['b','separates','a'])
assert check(counterexample)['reason'] == 'GOAL_UNMET'
print('Irreflexivity survives; the directed goal does not:', check(counterexample))

## 8. Learn from rejection

Reproduce a retained defect and include a valid control, so rejecting everything cannot pass.

In [ ]:
retained = deepcopy(example(['a','separates','a']))
first, replay = check(retained), check(deepcopy(retained))
assert first == replay and replay['reason'] == 'SELF_RELATION'
assert check(example())['status'] == 'CHECKED'
print('Counterexample replay:', replay)

## 9. Check an extension

Illustrative finite up/down maps only. This example licenses no Genesis or MSS transition.

In [ ]:
old_terms = ['a','b']
def up(mark):
    return ('old', mark)
def down(term):
    return term[1] if term[0] == 'old' and term[1] in old_terms else 'NON_DESCENT'
assert all(down(up(mark)) == mark for mark in old_terms)
assert down(('new','c')) == 'NON_DESCENT'
print('Old-image round trips pass; new term gives NON_DESCENT.')

## 10. Compare bounded strategies

Both strategies have the same triangle, goal, checker, and four-attempt budget. No global performance claim follows.

In [ ]:
records = []
for name, values in [('direct',[-3]),('incremental',[1,-1,-3])]:
    host = board(fuel=4)
    outcomes = [cut(host,value)['decision'] for value in values]
    receipt = host.execute_move({'move_type':'VERIFY'})
    assert receipt['decision'] == 'ACCEPTED_ZERO_DEFECT'
    records.append({'strategy':name,'attempts':4-host.fuel,'cuts':outcomes,'scope':receipt['receipt']['scope']})
assert [r['attempts'] for r in records] == [2,4]
print(records)